In [ ]:
# ============================================================
# MONTHLY BILLING VALIDATION — SAFE SETUP
# Uses monthly_consumption.csv + locked smart-meter clusters
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import kruskal, chi2_contingency

warnings.filterwarnings("ignore")

OUTPUT_DIR = "monthly_billing_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MONTHLY_FILE = "monthly_consumption.csv"
LOCKED_CLUSTER_FILE = "clean_fm_clusters.pkl"

CLUSTER_ORDER = [
    "Peak-Exposed",
    "Typical Household",
    "Heavy Continuous",
    "Daytime-Active"
]

CORE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]

def infer_archetype_name_map(feature_table, label_col="final_cluster"):
    """
    Infer stable archetype names from the locked feature profile.
    This avoids unsafe dependence on arbitrary KMeans numeric cluster IDs.
    """
    required = [
        label_col,
        "evening_peak_ratio",
        "night_baseload_ratio",
        "load_factor",
        "log_daily_kwh"
    ]

    missing = [c for c in required if c not in feature_table.columns]
    if missing:
        raise RuntimeError(
            "Cannot infer archetype names because required columns are missing: "
            + str(missing)
        )

    med = feature_table.groupby(label_col)[
        ["evening_peak_ratio", "night_baseload_ratio", "load_factor", "log_daily_kwh"]
    ].median()

    remaining = set(med.index.tolist())
    mapping = {}

    heavy_id = med["load_factor"].idxmax()
    mapping[heavy_id] = "Heavy Continuous"
    remaining.remove(heavy_id)

    peak_id = med.loc[list(remaining), "evening_peak_ratio"].idxmax()
    mapping[peak_id] = "Peak-Exposed"
    remaining.remove(peak_id)

    daytime_id = med.loc[list(remaining), "night_baseload_ratio"].idxmin()
    mapping[daytime_id] = "Daytime-Active"
    remaining.remove(daytime_id)

    typical_id = list(remaining)[0]
    mapping[typical_id] = "Typical Household"

    return dict(sorted(mapping.items(), key=lambda kv: kv[0]))

CLUSTER_NAMES = {}
CLUSTER_IDS = []
CLUSTER_NAME_TO_ID = {}

SMART_GOLDEN_START = pd.Timestamp("2024-05-01")
SMART_GOLDEN_END = pd.Timestamp("2024-12-01")  # exclusive upper bound for full May-Nov 2024

# ------------------------------------------------------------
# Load monthly billing data
# ------------------------------------------------------------
monthly_df = pd.read_csv(MONTHLY_FILE, low_memory=False)

required_monthly_cols = ["household_ID", "month", "consumption"]
missing_monthly = [c for c in required_monthly_cols if c not in monthly_df.columns]
if missing_monthly:
    raise RuntimeError(f"Missing monthly billing columns: {missing_monthly}")

monthly_df["household_ID"] = monthly_df["household_ID"].astype(str)

monthly_df["consumption"] = pd.to_numeric(
    monthly_df["consumption"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

monthly_df["month"] = pd.to_datetime(monthly_df["month"], errors="coerce")

monthly_df = monthly_df.dropna(subset=["household_ID", "month"]).copy()

# ------------------------------------------------------------
# Load locked smart-meter clusters
# ------------------------------------------------------------
clean_fm = pd.read_pickle(LOCKED_CLUSTER_FILE)
clean_fm.index = clean_fm.index.astype(str)

if "final_cluster" not in clean_fm.columns:
    raise RuntimeError("clean_fm_clusters.pkl must contain final_cluster.")

labels_df = clean_fm[["final_cluster"]].reset_index()

if "index" in labels_df.columns:
    labels_df = labels_df.rename(columns={"index": "household_ID"})

labels_df["household_ID"] = labels_df["household_ID"].astype(str)

CLUSTER_NAMES = infer_archetype_name_map(clean_fm, label_col="final_cluster")
CLUSTER_NAME_TO_ID = {v: k for k, v in CLUSTER_NAMES.items()}
CLUSTER_IDS = [CLUSTER_NAME_TO_ID[name] for name in CLUSTER_ORDER]

labels_df["cluster_name"] = labels_df["final_cluster"].map(CLUSTER_NAMES)

if labels_df["cluster_name"].isna().any():
    raise RuntimeError("Some final_cluster values could not be mapped to archetype names.")

print("Dynamic cluster-name map inferred from locked feature medians:")
for cluster_id, cluster_name in CLUSTER_NAMES.items():
    print(f"  {cluster_id} -> {cluster_name}")

# ------------------------------------------------------------
# Basic audit
# ------------------------------------------------------------
print("=" * 80)
print("MONTHLY BILLING DATA AUDIT")
print("=" * 80)

print("Monthly billing rows:", len(monthly_df))
print("Monthly billing households:", monthly_df["household_ID"].nunique())
print("Monthly date range:", monthly_df["month"].min(), "to", monthly_df["month"].max())
print("Distinct billing months:", monthly_df["month"].nunique())
print("Missing consumption values:", monthly_df["consumption"].isna().sum())

print("\nConsumption distribution:")
print(monthly_df["consumption"].describe().round(3).to_string())

months_per_household = monthly_df.groupby("household_ID")["month"].nunique()

print("\nMonths per billing household:")
print(months_per_household.describe().round(3).to_string())

# ------------------------------------------------------------
# Merge with locked clusters
# ------------------------------------------------------------
merged = monthly_df.merge(
    labels_df[["household_ID", "final_cluster", "cluster_name"]],
    on="household_ID",
    how="inner"
)

print("\nMerged monthly-billing + locked-cluster rows:", len(merged))
print("Matched locked-cluster households:", merged["household_ID"].nunique())

print("\nMatched cluster distribution:")
print(
    merged[["household_ID", "cluster_name"]]
    .drop_duplicates()
    ["cluster_name"]
    .value_counts()
    .reindex(CLUSTER_ORDER)
    .fillna(0)
    .astype(int)
    .to_string()
)

print("\nSafe interpretation:")
print(
    "Monthly billing data are used only as external validation for the locked "
    "smart-meter clusters. The billing-matched sample is a subset of the full "
    f"{len(labels_df):,}-household clustering population."
)

In [ ]:
# ============================================================
# FORMAL MONTHLY-BILLING MATCH-BIAS AUDIT
# Chi-square, Cramer's V, standardized residuals, Wilson CIs
# ============================================================

print("=" * 80)
print("FORMAL MONTHLY-BILLING MATCH-BIAS AUDIT")
print("=" * 80)

monthly_ids = set(monthly_df["household_ID"].astype(str).unique())

labels_match = labels_df.copy()
labels_match["matched_to_monthly"] = labels_match["household_ID"].isin(monthly_ids)

ct = pd.crosstab(
    labels_match["final_cluster"],
    labels_match["matched_to_monthly"]
).reindex(
    index=CLUSTER_IDS,
    columns=[False, True],
    fill_value=0
)

display_ct = ct.copy()
display_ct.index = [CLUSTER_NAMES[i] for i in display_ct.index]
display_ct.columns = ["Not matched", "Matched"]

print("Cluster × monthly-billing match status:")
print(display_ct.to_string())

chi2, p, dof, expected = chi2_contingency(ct)

n_total = ct.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n_total * (min(ct.shape) - 1)))

print(f"\nChi-square={chi2:.4f}, df={dof}, p={p:.8g}")
print(f"Cramer's V={cramers_v:.4f}")

observed_array = ct.to_numpy(dtype=float)
expected_array = np.asarray(expected, dtype=float)

pearson_residual_array = (
    observed_array - expected_array
) / np.sqrt(expected_array)

resid_df = pd.DataFrame(
    pearson_residual_array,
    index=[CLUSTER_NAMES[i] for i in ct.index],
    columns=["Not matched", "Matched"]
)

print("\nPearson residuals:")
print(resid_df.round(3).to_string())
print(
    "\nLarger absolute Pearson residuals identify cells "
    "contributing more strongly to the overall association."
)

def wilson_interval(successes, total, z=1.959963984540054):
    if total == 0:
        return np.nan, np.nan

    phat = successes / total
    denominator = 1 + z**2 / total

    centre = (phat + z**2 / (2 * total)) / denominator

    half_width = (
        z
        * np.sqrt(
            phat * (1 - phat) / total
            + z**2 / (4 * total**2)
        )
        / denominator
    )

    return centre - half_width, centre + half_width

rows = []

for cluster_id in CLUSTER_IDS:
    matched_n = int(ct.loc[cluster_id, True])
    total_n = int(ct.loc[cluster_id].sum())
    low, high = wilson_interval(matched_n, total_n)

    rows.append({
        "cluster": CLUSTER_NAMES[cluster_id],
        "matched_n": matched_n,
        "total_n": total_n,
        "match_rate_pct": matched_n / total_n * 100,
        "wilson_95_low_pct": low * 100,
        "wilson_95_high_pct": high * 100,
        "chi_square": chi2,
        "chi_square_df": dof,
        "chi_square_p": p,
        "cramers_v": cramers_v
    })

match_audit = pd.DataFrame(rows)

print("\nCluster-specific match rates with Wilson 95% confidence intervals:")
print(match_audit.round(2).to_string(index=False))

match_audit.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_match_bias_formal_audit.csv"),
    index=False
)

resid_df.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_match_bias_standardized_residuals.csv")
)

display_ct.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_match_status_table.csv")
)

print("\nSaved:")
print("- monthly_billing_outputs/monthly_billing_match_bias_formal_audit.csv")
print("- monthly_billing_outputs/monthly_billing_match_bias_pearson_residuals.csv")
print("- monthly_billing_outputs/monthly_billing_match_status_table.csv")

print("\nSafe interpretation:")
print(
    "Monthly billing availability differs significantly by cluster. "
    "Daytime-Active is strongly under-matched, so monthly billing results "
    "for that cluster must be interpreted cautiously. This audit does not "
    f"change the locked clustering labels or the N={len(labels_df):,} clustering cohort."
)

In [ ]:
# ============================================================
# FINAL HOUSEHOLD-LEVEL MONTHLY BILLING VALIDATION
# Corrected: household is the unit of analysis
# ============================================================

print("=" * 80)
print("FINAL HOUSEHOLD-LEVEL MONTHLY BILLING VALIDATION")
print("=" * 80)

# Monthly billing inside smart-meter golden window.
# The smart-meter golden window is May-Nov 2024, but billing file may only
# contain May-Oct 2024 depending on available monthly billing data.
billing_golden = merged.loc[
    (merged["month"] >= SMART_GOLDEN_START) &
    (merged["month"] < SMART_GOLDEN_END)
].copy()

available_golden_months = sorted(
    billing_golden["month"].dt.strftime("%Y-%m").dropna().unique()
)

print("Smart-meter golden window:", SMART_GOLDEN_START.date(), "to", (SMART_GOLDEN_END - pd.Timedelta(days=1)).date())
print("Available billing months inside this window:", available_golden_months)

# ------------------------------------------------------------
# Household-level aggregation
# ------------------------------------------------------------
hh_full = (
    merged
    .groupby(["household_ID", "final_cluster", "cluster_name"])
    ["consumption"]
    .median()
    .reset_index(name="median_kwh_full_panel")
)

hh_golden = (
    billing_golden
    .groupby(["household_ID", "final_cluster", "cluster_name"])
    ["consumption"]
    .median()
    .reset_index(name="median_kwh_billing_golden")
)

hh = hh_full.merge(
    hh_golden,
    on=["household_ID", "final_cluster", "cluster_name"],
    how="inner"
)

print("\nHouseholds with both full-panel and golden-window billing medians:", len(hh))

# ------------------------------------------------------------
# Kruskal-Wallis tests
# ------------------------------------------------------------
def run_kw_by_cluster(df, value_col, label):
    print("\n" + "-" * 80)
    print(label)
    print("-" * 80)

    groups = [
        df.loc[df["final_cluster"] == c, value_col].dropna().to_numpy()
        for c in CLUSTER_IDS
    ]

    stat, p = kruskal(*groups)

    print(f"Kruskal-Wallis H={stat:.3f}, p={p:.6g}")
    print(f"{'Cluster':<24} {'Median':>10} {'Mean':>10} {'N':>6}")
    print("-" * 54)

    for c in CLUSTER_IDS:
        vals = df.loc[df["final_cluster"] == c, value_col].dropna()
        print(
            f"{CLUSTER_NAMES[c]:<24} "
            f"{vals.median():>10.1f} "
            f"{vals.mean():>10.1f} "
            f"{len(vals):>6}"
        )

    return stat, p

stat_full, p_full = run_kw_by_cluster(
    hh,
    "median_kwh_full_panel",
    "Full billing panel household median"
)

stat_golden, p_golden = run_kw_by_cluster(
    hh,
    "median_kwh_billing_golden",
    "Golden-window billing household median"
)

# ------------------------------------------------------------
# Dunn post-hoc test if scikit-posthocs is available
# ------------------------------------------------------------
try:
    import scikit_posthocs as sp

    dunn = sp.posthoc_dunn(
        hh.dropna(subset=["median_kwh_billing_golden"]),
        val_col="median_kwh_billing_golden",
        group_col="final_cluster",
        p_adjust="bonferroni"
    )

    dunn.index = [CLUSTER_NAMES[int(i)] for i in dunn.index]
    dunn.columns = [CLUSTER_NAMES[int(i)] for i in dunn.columns]

    print("\nDunn post-hoc Bonferroni test for golden-window household median:")
    print(dunn.round(4).to_string())

    dunn.to_csv(
        os.path.join(OUTPUT_DIR, "monthly_billing_dunn_posthoc_golden_household_median.csv")
    )

except Exception as e:
    print("\nDunn post-hoc skipped because scikit_posthocs is unavailable or failed:")
    print(e)

# ------------------------------------------------------------
# Billing percentiles by cluster
# ------------------------------------------------------------
percentile_rows = []

for c in CLUSTER_IDS:
    vals = hh.loc[
        hh["final_cluster"] == c,
        "median_kwh_billing_golden"
    ].dropna()

    p10, p25, p50, p75, p90 = np.percentile(vals, [10, 25, 50, 75, 90])

    percentile_rows.append({
        "cluster": CLUSTER_NAMES[c],
        "n": len(vals),
        "p10": p10,
        "p25": p25,
        "p50_median": p50,
        "p75": p75,
        "p90": p90
    })

percentiles_df = pd.DataFrame(percentile_rows)

print("\nGolden-window household median billing percentiles:")
print(percentiles_df.round(1).to_string(index=False))

# ------------------------------------------------------------
# Volatility: CV per household over full billing panel
# ------------------------------------------------------------
hh_cv = (
    merged
    .groupby(["household_ID", "final_cluster", "cluster_name"])
    ["consumption"]
    .agg(["mean", "std"])
    .reset_index()
)

hh_cv["billing_cv"] = np.where(
    hh_cv["mean"] > 0,
    hh_cv["std"] / hh_cv["mean"],
    np.nan
)

cv_rows = []

for c in CLUSTER_IDS:
    vals = hh_cv.loc[hh_cv["final_cluster"] == c, "billing_cv"].dropna()

    cv_rows.append({
        "cluster": CLUSTER_NAMES[c],
        "n": len(vals),
        "median_cv": vals.median(),
        "mean_cv": vals.mean()
    })

cv_summary = pd.DataFrame(cv_rows)

print("\nBilling volatility by cluster:")
print(cv_summary.round(3).to_string(index=False))

groups_cv = [
    hh_cv.loc[hh_cv["final_cluster"] == c, "billing_cv"].dropna().to_numpy()
    for c in CLUSTER_IDS
]

stat_cv, p_cv = kruskal(*groups_cv)

print(f"\nKruskal-Wallis on billing CV: H={stat_cv:.3f}, p={p_cv:.6g}")

# ------------------------------------------------------------
# Near-zero months, household-level percentage
# ------------------------------------------------------------
near_zero = (
    merged
    .dropna(subset=["consumption"])
    .copy()
)

near_zero["near_zero_month"] = (
    near_zero["consumption"] < 10
)

hh_zero = (
    near_zero
    .groupby(["household_ID", "final_cluster", "cluster_name"])
    ["near_zero_month"]
    .mean()
    .reset_index(name="pct_near_zero_months")
)

hh_zero["pct_near_zero_months"] *= 100

zero_summary = (
    hh_zero
    .groupby("cluster_name")["pct_near_zero_months"]
    .agg(["count", "mean", "median"])
    .reset_index()
)

print("\nNear-zero billing months by cluster (% of months <10 kWh per household):")
print(zero_summary.round(2).to_string(index=False))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
hh.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_household_median_validation.csv"),
    index=False
)

percentiles_df.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_golden_percentiles_by_cluster.csv"),
    index=False
)

cv_summary.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_cv_by_cluster.csv"),
    index=False
)

zero_summary.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_near_zero_by_cluster.csv"),
    index=False
)

print("\nSaved:")
print("- monthly_billing_outputs/monthly_billing_household_median_validation.csv")
print("- monthly_billing_outputs/monthly_billing_golden_percentiles_by_cluster.csv")
print("- monthly_billing_outputs/monthly_billing_cv_by_cluster.csv")
print("- monthly_billing_outputs/monthly_billing_near_zero_by_cluster.csv")

print("\nSafe interpretation:")
print(
    "The locked smart-meter clusters show statistically significant differences "
    "in household-level monthly billing medians. Heavy Continuous has the highest "
    "billing consumption, while Peak-Exposed has the lowest. However, the billing "
    "validation is based only on the matched billing subset, and Daytime-Active "
    "has very small matched coverage, so that cluster must be interpreted cautiously."
)

In [ ]:
missing_ids = hh.loc[
    hh["median_kwh_billing_golden"].isna(),
    "household_ID"
]

print("Missing households:", missing_ids.tolist())

raw = pd.read_csv(MONTHLY_FILE, low_memory=False)
raw["household_ID"] = raw["household_ID"].astype(str)
raw["month"] = pd.to_datetime(raw["month"], errors="coerce")

check = raw.loc[
    raw["household_ID"].isin(missing_ids)
    & raw["month"].between("2024-05-01", "2024-10-31"),
    ["household_ID", "month", "consumption"]
].copy()

check["numeric_consumption"] = pd.to_numeric(
    check["consumption"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

print(check.sort_values("month").to_string(index=False))
print("Valid consumption values:", check["numeric_consumption"].notna().sum())

In [ ]:
# ============================================================
# EXPLORATORY TEMPORAL MONTHLY BILLING ANALYSIS
# Descriptive only; not causal
# ============================================================

print("=" * 80)
print("EXPLORATORY TEMPORAL MONTHLY BILLING ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Pre/post tariff-period descriptive comparison
# ------------------------------------------------------------
# This is descriptive. It should not be interpreted causally because tariff
# changes overlap with other time-varying factors and only matched households
# are included.

pre_period = merged.loc[merged["month"] < pd.Timestamp("2023-02-01")].copy()
post_period = merged.loc[merged["month"] >= pd.Timestamp("2023-08-01")].copy()

print("\n1. Pre/post tariff-period descriptive comparison")
print("Pre period: before 2023-02-01")
print("Post period: from 2023-08-01 onward")
print("Interpretation: descriptive only, not causal.")

period_summaries = []

for period_name, df_period in [
    ("pre_period", pre_period),
    ("post_period", post_period)
]:
    hh_period = (
        df_period
        .groupby(["household_ID", "final_cluster", "cluster_name"])
        ["consumption"]
        .median()
        .reset_index(name="median_kwh")
    )

    for c in CLUSTER_IDS:
        vals = hh_period.loc[hh_period["final_cluster"] == c, "median_kwh"].dropna()

        period_summaries.append({
            "period": period_name,
            "cluster": CLUSTER_NAMES[c],
            "n": len(vals),
            "median": vals.median(),
            "mean": vals.mean()
        })

period_summary_df = pd.DataFrame(period_summaries)

print("\nPre/post household median billing summary:")
print(period_summary_df.round(2).to_string(index=False))

pre_hh = (
    pre_period
    .groupby(["household_ID", "final_cluster", "cluster_name"])
    ["consumption"]
    .median()
    .reset_index(name="pre_median")
)

post_hh = (
    post_period
    .groupby(["household_ID", "final_cluster", "cluster_name"])
    ["consumption"]
    .median()
    .reset_index(name="post_median")
)

change = pre_hh.merge(
    post_hh,
    on=["household_ID", "final_cluster", "cluster_name"],
    how="inner"
)

change["delta_kwh"] = change["post_median"] - change["pre_median"]

change["pct_change"] = np.where(
    change["pre_median"] > 0,
    change["delta_kwh"] / change["pre_median"] * 100,
    np.nan
)

change_rows = []

for c in CLUSTER_IDS:
    sub = change.loc[change["final_cluster"] == c].copy()

    change_rows.append({
        "cluster": CLUSTER_NAMES[c],
        "n": len(sub),
        "median_delta_kwh": sub["delta_kwh"].median(),
        "median_pct_change": sub["pct_change"].median()
    })

change_summary = pd.DataFrame(change_rows)

print("\nHousehold-level pre/post change:")
print(change_summary.round(2).to_string(index=False))

groups_change = [
    change.loc[change["final_cluster"] == c, "pct_change"].dropna().to_numpy()
    for c in CLUSTER_IDS
]

stat_change, p_change = kruskal(*groups_change)

print(f"\nKruskal-Wallis on pre/post % change: H={stat_change:.3f}, p={p_change:.6g}")

# ------------------------------------------------------------
# 2. Month-by-month median billing by cluster
# ------------------------------------------------------------
ts = (
    merged
    .groupby(["month", "cluster_name"])
    ["consumption"]
    .median()
    .unstack()
)

ts = ts.reindex(columns=CLUSTER_ORDER)

print("\n2. Month-by-month median billing consumption by cluster:")
print(ts.round(1).to_string())

# ------------------------------------------------------------
# 3. Boundary hopper analysis: 120 kWh threshold
# ------------------------------------------------------------
hh_months = (
    merged
    .groupby(["household_ID", "final_cluster", "cluster_name"])
    .agg(
        months_above_120=("consumption", lambda x: (x > 120).sum()),
        total_months=("consumption", "count")
    )
    .reset_index()
)

hh_months["pct_months_above_120"] = (
    hh_months["months_above_120"] / hh_months["total_months"] * 100
)

def boundary_category(p):
    if p == 0:
        return "Never above 120"
    if p == 100:
        return "Always above 120"
    return "Boundary hopper"

hh_months["boundary_category"] = hh_months["pct_months_above_120"].apply(boundary_category)

boundary_table = (
    pd.crosstab(
        hh_months["cluster_name"],
        hh_months["boundary_category"],
        normalize="index"
    ) * 100
).reindex(index=CLUSTER_ORDER)

print("\n3. Boundary-hopper categories by cluster (% within cluster):")
print(boundary_table.round(1).to_string())

boundary_median = (
    hh_months
    .groupby("cluster_name")["pct_months_above_120"]
    .agg(["count", "median", "mean"])
    .reset_index()
)

print("\nMedian % of months above 120 kWh:")
print(boundary_median.round(2).to_string(index=False))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
period_summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_pre_post_period_summary.csv"),
    index=False
)

change_summary.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_pre_post_change_summary.csv"),
    index=False
)

change.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_pre_post_household_change.csv"),
    index=False
)

ts.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_monthly_median_timeseries_by_cluster.csv")
)

hh_months.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_120kwh_boundary_hopper_households.csv"),
    index=False
)

boundary_table.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_120kwh_boundary_category_percent.csv")
)

print("\nSaved:")
print("- monthly_billing_outputs/monthly_billing_pre_post_period_summary.csv")
print("- monthly_billing_outputs/monthly_billing_pre_post_change_summary.csv")
print("- monthly_billing_outputs/monthly_billing_pre_post_household_change.csv")
print("- monthly_billing_outputs/monthly_billing_monthly_median_timeseries_by_cluster.csv")
print("- monthly_billing_outputs/monthly_billing_120kwh_boundary_hopper_households.csv")
print("- monthly_billing_outputs/monthly_billing_120kwh_boundary_category_percent.csv")

print("\nSafe interpretation:")
print(
    "This temporal analysis is exploratory. It describes billing changes and "
    "threshold movement among matched households, but it should not be interpreted "
    "as causal evidence of tariff response."
)

In [ ]:
# ============================================================
# OPTIONAL EXPLORATORY TARIFF / AFFORDABILITY ANALYSIS
# Safe wording: simplified estimates, not official bills or causal results
# ============================================================

print("=" * 80)
print("OPTIONAL EXPLORATORY TARIFF / AFFORDABILITY ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Tariff bracket distribution using household golden-window median
# ------------------------------------------------------------
def tariff_bracket(units):
    if pd.isna(units):
        return np.nan
    if units <= 30:
        return "0-30"
    if units <= 60:
        return "31-60"
    if units <= 90:
        return "61-90"
    if units <= 120:
        return "91-120"
    if units <= 180:
        return "121-180"
    if units <= 240:
        return "181-240"
    return "240+"

hh_tariff = hh.copy()
hh_tariff["tariff_bracket_by_median_kwh"] = hh_tariff["median_kwh_billing_golden"].apply(tariff_bracket)

tariff_table = (
    pd.crosstab(
        hh_tariff["cluster_name"],
        hh_tariff["tariff_bracket_by_median_kwh"],
        normalize="index"
    ) * 100
).reindex(index=CLUSTER_ORDER)

print("\nTariff bracket distribution by cluster using household golden-window median kWh:")
print(tariff_table.round(1).to_string())

# ------------------------------------------------------------
# 2. Simplified variable-charge estimate
# ------------------------------------------------------------
# This is a simplified block-rate estimate using the rates below.
# It excludes fixed charges, taxes, subsidies, and period-specific tariff changes.
# Replace these values if your final methodology uses a specific official schedule.

def estimate_simplified_variable_charge(units):
    if pd.isna(units) or units <= 0:
        return 0.0

    u = float(units)

    if u <= 30:
        return u * 2.50
    if u <= 60:
        return 30 * 2.50 + (u - 30) * 4.85
    if u <= 90:
        return 30 * 2.50 + 30 * 4.85 + (u - 60) * 8.55
    if u <= 120:
        return 30 * 2.50 + 30 * 4.85 + 30 * 8.55 + (u - 90) * 15.45
    if u <= 180:
        return 30 * 2.50 + 30 * 4.85 + 30 * 8.55 + 30 * 15.45 + (u - 120) * 26.50
    if u <= 240:
        return (
            30 * 2.50 + 30 * 4.85 + 30 * 8.55 + 30 * 15.45
            + 60 * 26.50 + (u - 180) * 45.00
        )

    return (
        30 * 2.50 + 30 * 4.85 + 30 * 8.55 + 30 * 15.45
        + 60 * 26.50 + 60 * 45.00 + (u - 240) * 55.00
    )

billing_golden_charge = billing_golden.copy()

billing_golden_charge["simplified_variable_charge_lkr"] = (
    billing_golden_charge["consumption"]
    .apply(estimate_simplified_variable_charge)
)

charge_summary = (
    billing_golden_charge
    .groupby("cluster_name")
    .agg(
        n_rows=("simplified_variable_charge_lkr", "count"),
        median_charge=("simplified_variable_charge_lkr", "median"),
        mean_charge=("simplified_variable_charge_lkr", "mean"),
        total_simplified_charge=("simplified_variable_charge_lkr", "sum")
    )
    .reset_index()
)

charge_summary["share_of_simplified_charge_percent"] = (
    charge_summary["total_simplified_charge"] /
    charge_summary["total_simplified_charge"].sum() * 100
)

print("\nSimplified variable-charge estimate by cluster:")
print(charge_summary.round(2).to_string(index=False))

# ------------------------------------------------------------
# 3. Energy intensity using Wave 1 expenditure
# ------------------------------------------------------------
hh_info_path = "wave_1/w1_household_information_and_history.csv"

if os.path.exists(hh_info_path):
    hh_info = pd.read_csv(hh_info_path, low_memory=False)

    required_cols = ["household_ID", "total_monthly_expenditure_of_last_month"]

    if all(c in hh_info.columns for c in required_cols):
        exp = hh_info[required_cols].copy()
        exp["household_ID"] = exp["household_ID"].astype(str)
        exp["total_monthly_expenditure_of_last_month"] = pd.to_numeric(
            exp["total_monthly_expenditure_of_last_month"],
            errors="coerce"
        )

        intensity = hh.merge(
            exp,
            on="household_ID",
            how="inner"
        )

        intensity["kwh_per_1000_lkr_expenditure"] = np.where(
            intensity["total_monthly_expenditure_of_last_month"] > 0,
            intensity["median_kwh_billing_golden"] /
            intensity["total_monthly_expenditure_of_last_month"] * 1000,
            np.nan
        )

        intensity_rows = []

        for c in CLUSTER_IDS:
            vals = intensity.loc[
                intensity["final_cluster"] == c,
                "kwh_per_1000_lkr_expenditure"
            ].replace([np.inf, -np.inf], np.nan).dropna()

            intensity_rows.append({
                "cluster": CLUSTER_NAMES[c],
                "n": len(vals),
                "median_kwh_per_1000_lkr": vals.median(),
                "mean_kwh_per_1000_lkr": vals.mean()
            })

        intensity_summary = pd.DataFrame(intensity_rows)

        print("\nExploratory energy intensity by cluster:")
        print(intensity_summary.round(3).to_string(index=False))

        groups_intensity = [
            intensity.loc[
                intensity["final_cluster"] == c,
                "kwh_per_1000_lkr_expenditure"
            ].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
            for c in CLUSTER_IDS
        ]

        stat_intensity, p_intensity = kruskal(*groups_intensity)

        print(
            f"\nKruskal-Wallis on energy intensity: "
            f"H={stat_intensity:.3f}, p={p_intensity:.6g}"
        )

        intensity_summary.to_csv(
            os.path.join(OUTPUT_DIR, "monthly_billing_energy_intensity_summary.csv"),
            index=False
        )

        intensity.to_csv(
            os.path.join(OUTPUT_DIR, "monthly_billing_energy_intensity_households.csv"),
            index=False
        )

    else:
        print("\nWave 1 expenditure columns not found. Skipping energy intensity.")

else:
    print("\nWave 1 household information file not found. Skipping energy intensity.")

# ------------------------------------------------------------
# Save tariff outputs
# ------------------------------------------------------------
tariff_table.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_tariff_bracket_distribution.csv")
)

charge_summary.to_csv(
    os.path.join(OUTPUT_DIR, "monthly_billing_simplified_variable_charge_summary.csv"),
    index=False
)

print("\nSaved:")
print("- monthly_billing_outputs/monthly_billing_tariff_bracket_distribution.csv")
print("- monthly_billing_outputs/monthly_billing_simplified_variable_charge_summary.csv")
print("- monthly_billing_outputs/monthly_billing_energy_intensity_summary.csv, if available")
print("- monthly_billing_outputs/monthly_billing_energy_intensity_households.csv, if available")

print("\nSafe interpretation:")
print(
    "This section is exploratory. Tariff brackets are based on household median "
    "monthly billing consumption. The variable-charge estimate is simplified and "
    "should not be reported as an official bill or actual CEB revenue because "
    "fixed charges, taxes, subsidies, and time-specific tariff schedules are not "
    "fully modelled. Energy intensity combines billing data with Wave 1 expenditure "
    "and should be interpreted descriptively."
)

In [ ]:
# ============================================================
# OPTIONAL EXPLORATORY SEASONAL / MONTH-OF-YEAR BILLING PATTERN
# Descriptive only; not final inferential evidence
# ============================================================

print("=" * 80)
print("OPTIONAL EXPLORATORY SEASONAL / MONTH-OF-YEAR BILLING PATTERN")
print("=" * 80)

seasonal_df = merged.copy()

seasonal_df["month_number"] = seasonal_df["month"].dt.month
seasonal_df["month_name"] = seasonal_df["month"].dt.strftime("%b")

# Sri Lanka broad climate-season grouping.
# Used only for descriptive comparison, not causal explanation.
def sri_lanka_season(month_number):
    if month_number in [12, 1, 2]:
        return "Northeast monsoon / dry-intermediate"
    elif month_number in [3, 4]:
        return "First inter-monsoon"
    elif month_number in [5, 6, 7, 8, 9]:
        return "Southwest monsoon"
    elif month_number in [10, 11]:
        return "Second inter-monsoon"
    else:
        return "Unknown"

seasonal_df["season_group"] = seasonal_df["month_number"].apply(sri_lanka_season)

# ------------------------------------------------------------
# 1. Month-of-year median billing by cluster
# ------------------------------------------------------------
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

month_pattern = (
    seasonal_df
    .groupby(["month_number", "month_name", "cluster_name"])
    ["consumption"]
    .median()
    .reset_index()
)

month_pattern_pivot = (
    month_pattern
    .pivot_table(
        index=["month_number", "month_name"],
        columns="cluster_name",
        values="consumption"
    )
    .reset_index()
    .sort_values("month_number")
)

month_pattern_pivot = month_pattern_pivot[
    ["month_number", "month_name"] + CLUSTER_ORDER
]

print("\nMonth-of-year median billing consumption by cluster:")
print(month_pattern_pivot.round(1).to_string(index=False))

# ------------------------------------------------------------
# 2. Season-group median billing by cluster
# ------------------------------------------------------------
season_pattern = (
    seasonal_df
    .groupby(["season_group", "cluster_name"])
    ["consumption"]
    .median()
    .reset_index()
)

season_pattern_pivot = (
    season_pattern
    .pivot_table(
        index="season_group",
        columns="cluster_name",
        values="consumption"
    )
    .reindex(columns=CLUSTER_ORDER)
)

print("\nSeason-group median billing consumption by cluster:")
print(season_pattern_pivot.round(1).to_string())

# ------------------------------------------------------------
# 3. Household-level seasonal medians
# Correct unit: household-season, not raw repeated readings
# ------------------------------------------------------------
hh_season = (
    seasonal_df
    .groupby(["household_ID", "final_cluster", "cluster_name", "season_group"])
    ["consumption"]
    .median()
    .reset_index(name="household_season_median_kwh")
)

hh_season_summary = (
    hh_season
    .groupby(["season_group", "cluster_name"])
    ["household_season_median_kwh"]
    .agg(["count", "median", "mean"])
    .reset_index()
)

print("\nHousehold-level seasonal median billing summary:")
print(hh_season_summary.round(2).to_string(index=False))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
month_pattern_pivot.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_month_of_year_billing_pattern_by_cluster.csv"),
    index=False
)

season_pattern_pivot.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_season_group_billing_pattern_by_cluster.csv")
)

hh_season_summary.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_household_level_seasonal_billing_summary.csv"),
    index=False
)

hh_season.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_household_season_median_billing.csv"),
    index=False
)

print("\nSaved:")
print("- monthly_billing_outputs/exploratory_month_of_year_billing_pattern_by_cluster.csv")
print("- monthly_billing_outputs/exploratory_season_group_billing_pattern_by_cluster.csv")
print("- monthly_billing_outputs/exploratory_household_level_seasonal_billing_summary.csv")
print("- monthly_billing_outputs/exploratory_household_season_median_billing.csv")

print("\nSafe interpretation:")
print(
    "This seasonal/month-of-year section is exploratory only. It describes "
    "billing patterns across calendar months and broad Sri Lankan climate-season "
    "groups among the monthly-billing matched households. It should not be "
    "interpreted as causal weather, tariff, or behavioural evidence."
)

In [ ]:
# ============================================================
# OPTIONAL EXPLORATORY YEAR-ON-YEAR BILLING COMPARISON
# 2023 vs 2024 matched-month comparison
# Descriptive only; not final inferential evidence
# ============================================================

print("=" * 80)
print("OPTIONAL EXPLORATORY YEAR-ON-YEAR BILLING COMPARISON")
print("=" * 80)

yoy_df = merged.copy()

yoy_df["year"] = yoy_df["month"].dt.year
yoy_df["month_number"] = yoy_df["month"].dt.month

# Compare only months available in both 2023 and 2024.
months_2023 = set(
    yoy_df.loc[yoy_df["year"] == 2023, "month_number"]
    .dropna()
    .astype(int)
    .unique()
)

months_2024 = set(
    yoy_df.loc[yoy_df["year"] == 2024, "month_number"]
    .dropna()
    .astype(int)
    .unique()
)

common_months = sorted(months_2023.intersection(months_2024))

print("Common months available in both 2023 and 2024:")
print(common_months)

yoy_common = yoy_df.loc[
    yoy_df["year"].isin([2023, 2024]) &
    yoy_df["month_number"].isin(common_months)
].copy()

# ------------------------------------------------------------
# 1. Cluster-level monthly median table
# ------------------------------------------------------------
yoy_monthly_cluster = (
    yoy_common
    .groupby(["year", "month_number", "cluster_name"])
    ["consumption"]
    .median()
    .reset_index()
)

yoy_monthly_pivot = (
    yoy_monthly_cluster
    .pivot_table(
        index=["month_number", "cluster_name"],
        columns="year",
        values="consumption"
    )
    .reset_index()
)

if 2023 in yoy_monthly_pivot.columns and 2024 in yoy_monthly_pivot.columns:
    yoy_monthly_pivot["delta_2024_minus_2023"] = (
        yoy_monthly_pivot[2024] - yoy_monthly_pivot[2023]
    )

    yoy_monthly_pivot["pct_change_2024_vs_2023"] = np.where(
        yoy_monthly_pivot[2023] > 0,
        yoy_monthly_pivot["delta_2024_minus_2023"] / yoy_monthly_pivot[2023] * 100,
        np.nan
    )

print("\nMonth-matched cluster median billing: 2023 vs 2024")
print(yoy_monthly_pivot.round(2).to_string(index=False))

# ------------------------------------------------------------
# 2. Household-level YoY comparison
# Correct unit: household, not monthly rows
# ------------------------------------------------------------
hh_yoy = (
    yoy_common
    .groupby(["household_ID", "final_cluster", "cluster_name", "year"])
    ["consumption"]
    .median()
    .reset_index(name="year_median_kwh")
)

hh_yoy_pivot = (
    hh_yoy
    .pivot_table(
        index=["household_ID", "final_cluster", "cluster_name"],
        columns="year",
        values="year_median_kwh"
    )
    .reset_index()
)

if 2023 not in hh_yoy_pivot.columns or 2024 not in hh_yoy_pivot.columns:
    raise RuntimeError("Cannot compute YoY because 2023 or 2024 household medians are missing.")

hh_yoy_pivot = hh_yoy_pivot.dropna(subset=[2023, 2024]).copy()

hh_yoy_pivot["delta_2024_minus_2023"] = (
    hh_yoy_pivot[2024] - hh_yoy_pivot[2023]
)

hh_yoy_pivot["pct_change_2024_vs_2023"] = np.where(
    hh_yoy_pivot[2023] > 0,
    hh_yoy_pivot["delta_2024_minus_2023"] / hh_yoy_pivot[2023] * 100,
    np.nan
)

yoy_household_summary = (
    hh_yoy_pivot
    .groupby("cluster_name")
    .agg(
        n=("household_ID", "count"),
        median_2023=(2023, "median"),
        median_2024=(2024, "median"),
        median_delta_kwh=("delta_2024_minus_2023", "median"),
        median_pct_change=("pct_change_2024_vs_2023", "median"),
        mean_pct_change=("pct_change_2024_vs_2023", "mean")
    )
    .reset_index()
)

print("\nHousehold-level YoY billing comparison by locked cluster:")
print(yoy_household_summary.round(2).to_string(index=False))

# ------------------------------------------------------------
# 3. Optional Kruskal-Wallis test on household-level YoY % change
# Exploratory only
# ------------------------------------------------------------
groups_yoy = [
    hh_yoy_pivot.loc[
        hh_yoy_pivot["final_cluster"] == c,
        "pct_change_2024_vs_2023"
    ].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    for c in CLUSTER_IDS
]

valid_groups_yoy = [g for g in groups_yoy if len(g) > 0]

if len(valid_groups_yoy) >= 2:
    stat_yoy, p_yoy = kruskal(*valid_groups_yoy)
    print(
        f"\nExploratory Kruskal-Wallis on household-level YoY % change: "
        f"H={stat_yoy:.3f}, p={p_yoy:.6g}"
    )
else:
    print("\nYoY Kruskal-Wallis skipped because fewer than two valid cluster groups exist.")

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
yoy_monthly_pivot.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_yoy_month_matched_cluster_medians.csv"),
    index=False
)

hh_yoy_pivot.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_yoy_household_level_changes.csv"),
    index=False
)

yoy_household_summary.to_csv(
    os.path.join(OUTPUT_DIR, "exploratory_yoy_household_level_summary_by_cluster.csv"),
    index=False
)

print("\nSaved:")
print("- monthly_billing_outputs/exploratory_yoy_month_matched_cluster_medians.csv")
print("- monthly_billing_outputs/exploratory_yoy_household_level_changes.csv")
print("- monthly_billing_outputs/exploratory_yoy_household_level_summary_by_cluster.csv")

print("\nSafe interpretation:")
print(
    "This YoY section is exploratory only. It compares matched months in 2023 "
    "and 2024 among billing-matched households. It should not be interpreted "
    "as causal evidence of tariff response, climate effects, or behavioural "
    "change because other time-varying factors are not controlled."
)